# MCP Hands-On Seminar with GigaChat

This notebook is execution-first: run cells to see working agent behavior.
No source-code dump cells, only runnable functionality.


## Learning Objectives

By the end of this notebook you will be able to:
1. Run an MCP server locally with `stdio` transport.
2. Connect a GigaChat-based agent to MCP tools.
3. Run MCP via HTTP/SSE transport.
4. Use a live interactive chat loop via `input()` in a notebook cell.


In [17]:
from pathlib import Path
import json

MCP_DIR = Path('Function_Calling_MCP_seminar/MCP')
if not MCP_DIR.exists():
    MCP_DIR = Path('.')

print('Using MCP directory:', MCP_DIR.resolve())
print('Available files:')
for file_path in sorted(MCP_DIR.glob('*')):
    print('-', file_path.name)


Using MCP directory: /Users/daniilserki/Documents/PIL_course/prompt_engineering_seminar/Function_Calling_MCP_seminar/MCP
Available files:
- .env.example
- MCP_seminar_demo.ipynb
- README.md
- __pycache__
- agent.py
- agent_http.py
- math_server.py
- mcp_config.json
- mcp_react_agent.py
- requirements.txt


## Prerequisites

1. Set credentials in `.env`:
- `GIGACHAT_CREDENTIALS=...` or `API_KEY=...`

2. Install dependencies from MCP folder:


In [18]:
# Run once if needed
# !pip install -r requirements.txt


In [19]:
import os
import re
import sys
import time
import asyncio
import subprocess

import httpx
from dotenv import load_dotenv, find_dotenv
from langchain_gigachat import GigaChat
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

load_dotenv(find_dotenv(usecwd=True))


True

In [20]:
def resolve_gigachat_credentials() -> str:
    credentials = os.getenv('GIGACHAT_CREDENTIALS') or os.getenv('API_KEY')
    if not credentials:
        raise ValueError('Set GIGACHAT_CREDENTIALS or API_KEY in .env before running this notebook.')
    return credentials


def build_model() -> GigaChat:
    return GigaChat(
        model='GigaChat-2-Max',
        credentials=resolve_gigachat_credentials(),
        verify_ssl_certs=False,
        streaming=False,
        max_tokens=8000,
        timeout=600,
    )


def print_messages(messages):
    for message in messages:
        print(f"[{type(message).__name__}] {getattr(message, 'content', '')}")
        tool_calls = getattr(message, 'tool_calls', None)
        if tool_calls:
            for tool_call in tool_calls:
                print(f"  🔧 Tool: {tool_call['name']} | Args: {tool_call['args']}")


## 1) Local MCP with `stdio` (integrated `agent.py` logic)


In [21]:
async def run_stdio_demo(prompts):
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(MCP_DIR / 'math_server.py')],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            agent = create_react_agent(build_model(), tools)

            for prompt in prompts:
                response = await agent.ainvoke({'messages': [{'role': 'user', 'content': prompt}]})
                print()
                print(f"User: {prompt}")
                print_messages(response['messages'])
                print('-' * 80)


In [22]:
await run_stdio_demo([
    'What is (3 + 5) * 12?',
    'How old is John Doe?',
])


/var/folders/p0/jjpv1rtx7ljfjllbr544_xqr0000gn/T/ipykernel_10438/1874305901.py:11: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(build_model(), tools)



User: What is (3 + 5) * 12?
[HumanMessage] What is (3 + 5) * 12?
[AIMessage] 
  🔧 Tool: add | Args: {'a': 3, 'b': 5}
[ToolMessage] 8.0
[AIMessage] 
  🔧 Tool: multiply | Args: {'a': 8, 'b': 12}
[ToolMessage] 96.0
[AIMessage] The result of $(3 + 5) \times 12$ is $96$.
--------------------------------------------------------------------------------

User: How old is John Doe?
[HumanMessage] How old is John Doe?
[AIMessage] 
  🔧 Tool: find_preson | Args: {'name': {'query': 'John Doe'}}
[ToolMessage] {
  "name": "John Doe",
  "age": 30
}
[AIMessage] John Doe is 30 years old.
--------------------------------------------------------------------------------


## 2) MCP over HTTP/SSE (integrated `agent_http.py` logic)


In [23]:
def build_httpx_client(headers=None, timeout=None, auth=None):
    # trust_env=False avoids proxy interference for local SSE URLs.
    kwargs = {
        'follow_redirects': True,
        'trust_env': False,
        'timeout': timeout or httpx.Timeout(30.0, read=300.0),
    }
    if headers is not None:
        kwargs['headers'] = headers
    if auth is not None:
        kwargs['auth'] = auth
    return httpx.AsyncClient(**kwargs)


async def run_http_demo(prompts):
    client = MultiServerMCPClient(
        {
            'math': {
                'url': 'http://127.0.0.1:8000/sse',
                'transport': 'sse',
                'httpx_client_factory': build_httpx_client,
            }
        }
    )
    tools = await client.get_tools()
    agent = create_react_agent(build_model(), tools)

    for prompt in prompts:
        response = await agent.ainvoke({'messages': [{'role': 'user', 'content': prompt}]})
        print()
        print(f"User: {prompt}")
        print_messages(response['messages'])
        print('-' * 80)


In [24]:
sse_server = subprocess.Popen(
    [sys.executable, 'math_server.py', 'sse'],
    cwd=str(MCP_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(2)

try:
    await run_http_demo([
        'What is (3 + 5) * 12?',
        'How old is John Doe?',
    ])
finally:
    if sse_server.poll() is None:
        sse_server.terminate()
        try:
            sse_server.wait(timeout=5)
        except subprocess.TimeoutExpired:
            sse_server.kill()
    print('SSE server stopped.')


/var/folders/p0/jjpv1rtx7ljfjllbr544_xqr0000gn/T/ipykernel_10438/140687699.py:26: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(build_model(), tools)



User: What is (3 + 5) * 12?
[HumanMessage] What is (3 + 5) * 12?
[AIMessage] 
  🔧 Tool: add | Args: {'a': 3, 'b': 5}
[ToolMessage] 8.0
[AIMessage] 
  🔧 Tool: multiply | Args: {'a': 8, 'b': 12}
[ToolMessage] 96.0
[AIMessage] The result of $(3 + 5) \times 12$ is $96$.
--------------------------------------------------------------------------------

User: How old is John Doe?
[HumanMessage] How old is John Doe?
[AIMessage] 
  🔧 Tool: find_preson | Args: {'name': {'query': 'John Doe'}}
[ToolMessage] {
  "name": "John Doe",
  "age": 30
}
[AIMessage] John Doe is 30 years old.
--------------------------------------------------------------------------------
SSE server stopped.


## 3) Live Interactive Chat via `input()` (notebook cell)

This is the notebook-native dialog mode. Type `quit` (or empty line) to stop.


In [25]:
async def chat_with_agent_input(thread_id='notebook-chat-1'):
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(MCP_DIR / 'math_server.py')],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)

            agent = create_react_agent(
                build_model(),
                tools=tools,
                prompt='You are a helpful assistant.',
                checkpointer=MemorySaver(),
            )

            print("Interactive session started. Type 'quit' to stop.")
            while True:
                user_input = input('User: ').strip()
                if user_input.lower() in {'quit', 'exit', ''}:
                    print('Session finished.')
                    break

                response = await agent.ainvoke(
                    {'messages': [{'role': 'user', 'content': user_input}]},
                    config={'configurable': {'thread_id': thread_id}},
                )

                for msg in response['messages']:
                    if hasattr(msg, 'tool_calls') and msg.tool_calls:
                        for tool_call in msg.tool_calls:
                            print(f"🔧 Tool: {tool_call['name']} | Args: {tool_call['args']}")

                print('Agent:', response['messages'][-1].content)


In [26]:
# Run this cell for a live dialog.
# Stop with: quit
await chat_with_agent_input()


Interactive session started. Type 'quit' to stop.


/var/folders/p0/jjpv1rtx7ljfjllbr544_xqr0000gn/T/ipykernel_10438/1730770493.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


Agent: Firstly, we need to compute the sum inside the parentheses:

$$
12 + 1 = 13
$$

Then multiply this result by 2:

$$
13 \times 2 = 26
$$

Final answer:
The computed price is $26.
Agent: In your previous message, you mentioned that "I am like chips." This means you're comparing yourself to chips, possibly indicating that you're fun, easy-going, or enjoyable just as snacking on chips can be pleasurable and satisfying.

If you'd like further clarification or additional context, please provide more details!
Session finished.


## 4) Functional Diagnostics


In [ ]:
config_path = MCP_DIR / 'mcp_config.json'
config = json.loads(config_path.read_text(encoding='utf-8'))

print('Configured MCP servers:', ', '.join(config.keys()))
for name, cfg in config.items():
    print(f"- {name}: transport={cfg.get('transport')} command={cfg.get('command')} args={cfg.get('args')}")


Configured MCP servers: math
- math: transport=stdio command=python args=['math_server.py']


In [16]:
math_server_text = (MCP_DIR / 'math_server.py').read_text(encoding='utf-8')
functions = re.findall(r'def\s+([a-zA-Z_][a-zA-Z0-9_]*)\s*\(', math_server_text)

print('Detected callable functions in math_server.py:')
for fn in functions:
    print('-', fn)


Detected callable functions in math_server.py:
- find_preson
- add
- multiply


## Exercise 1: Add a New MCP Tool

Task:
1. Open `math_server.py`.
2. Add a new tool `subtract(a: float, b: float) -> float`.
3. Re-run the stdio demo cell and ask: "What is (10 - 3) * 5?"

Success criteria:
- The agent uses `subtract` before `multiply`.


## Exercise 2: Improve Tool Description Quality

Task:
1. Improve docstrings for `find_preson`, `add`, `multiply`.
2. Make descriptions explicit about parameter semantics.
3. Compare tool-call reliability before vs after edits.

Success criteria:
- Fewer wrong tool calls on ambiguous prompts.


## Exercise 3: Production-Style MCP Agent

Build a mini assistant using the `input()` chat cell:
1. Add one new tool to the MCP server.
2. Update `mcp_config.json` if needed.
3. Test with 5 multi-step prompts.
4. Record one failure case and fix it.

Deliverables:
- Updated code,
- test prompts,
- short failure analysis.


## Debrief

1. Which prompt patterns most reliably trigger the right MCP tool?
2. Where did tool routing fail: schema, description, or model reasoning?
3. What changes gave the highest reliability improvement?
